In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ReadS3Data") \
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262"
    ) \
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.DefaultAWSCredentialsProviderChain"
    ) \
    .getOrCreate()

26/03/24 08:05:03 WARN Utils: Your hostname, Abhyudays-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.1.2 instead (on interface en0)
26/03/24 08:05:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/opt/homebrew/Cellar/apache-spark/3.5.4/libexec/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/abhyudaysahay/.ivy2/cache
The jars for the packages stored in: /Users/abhyudaysahay/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3c002d1b-993a-4dde-b5f8-93866a0b2a44;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 435ms :: artifacts dl 23ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| se

Loading cleaned data

In [2]:
df = spark.read.parquet('s3a://de-project-staging-abhi/sales/')
df.show()

26/03/24 08:07:27 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


+------+--------------+----------+----------+--------------+-----------+------------------+---------+-------------+-------------+----------+-----------+---------------+---------------+------------+--------------------+--------+------+
|row_id|      order_id|order_date| ship_date|     ship_mode|customer_id|     customer_name|  segment|      country|         city|     state|postal_code|     product_id|       category|sub_category|        product_name|   sales|region|
+------+--------------+----------+----------+--------------+-----------+------------------+---------+-------------+-------------+----------+-----------+---------------+---------------+------------+--------------------+--------+------+
|     3|CA-2017-138688|2017-06-12|2017-06-16|  Second Class|   DV-13045|   Darrin Van Huff|Corporate|United States|  Los Angeles|California|      90036|OFF-LA-10000240|Office Supplies|      Labels|Self-Adhesive Add...|   14.62|  West|
|     6|CA-2015-115812|2015-06-09|2015-06-14|Standard Class|

Calculating total sales by region

In [3]:
from pyspark.sql.functions import sum

sales_by_region = df.groupBy('region') \
    .agg(sum('sales').alias('total_sales'))

sales_by_region.show()

+-------+-----------------+
| region|      total_sales|
+-------+-----------------+
|   West|710219.6845000003|
|   East|669518.7259999984|
|Central|492646.9132000005|
|  South|389151.4590000004|
+-------+-----------------+



In [11]:
#loding data to s3

sales_by_region.write \
    .mode('overwrite') \
    .parquet('s3a://de-project-curated-abhi/sales_by_region/')

Sales for each catagory and sub-catagory

In [4]:
sales_by_catagory = df.groupBy('category', 'sub_category') \
    .agg(sum('sales').alias('total_sales'))

sales_by_catagory.show()

+---------------+------------+------------------+
|       category|sub_category|       total_sales|
+---------------+------------+------------------+
|Office Supplies|  Appliances|104618.40299999998|
|Office Supplies|   Envelopes|16128.046000000002|
|      Furniture|      Chairs| 322822.7309999998|
|Office Supplies|     Storage|        219343.392|
|Office Supplies|       Paper| 76828.30399999997|
|Office Supplies|    Supplies|         46420.308|
|     Technology|     Copiers|        146248.094|
|Office Supplies|         Art|26705.410000000007|
|Office Supplies|     Binders|200028.78499999997|
|     Technology| Accessories|          164186.7|
|Office Supplies|   Fasteners|           3001.96|
|      Furniture|   Bookcases|       113813.1987|
|     Technology|      Phones|        327782.448|
|Office Supplies|      Labels|         12347.726|
|     Technology|    Machines|189238.63100000002|
|      Furniture|      Tables|202810.62799999997|
|      Furniture| Furnishings| 89212.01799999997|


In [12]:
#loding data to s3

sales_by_catagory.write \
    .mode('overwrite') \
    .parquet('s3a://de-project-curated-abhi/sales_by_catagory/')

Monthly sales trend

In [6]:
from pyspark.sql.functions import year, month

df_time = df \
    .withColumn('year', year('order_date')) \
    .withColumn('month', month('order_date'))

df_monthly_sales = df_time.groupBy('year', 'month') \
    .agg(sum('sales').alias('monthly_sales')) \
    .orderBy('year', 'month')

df_monthly_sales.show()

+----+-----+------------------+
|year|month|     monthly_sales|
+----+-----+------------------+
|2015|    1|14205.706999999999|
|2015|    2| 4519.892000000001|
|2015|    3| 55205.79699999998|
|2015|    4|27906.854999999996|
|2015|    5|         23644.303|
|2015|    6| 34322.93559999999|
|2015|    7|33781.543000000005|
|2015|    8|27117.536499999995|
|2015|    9| 81623.52679999999|
|2015|   10|31453.392999999996|
|2015|   11|        77907.6607|
|2015|   12|        68167.0585|
|2016|    1|        18066.9576|
|2016|    2|         11951.411|
|2016|    3|32339.318400000004|
|2016|    4|        34154.4685|
|2016|    5|29959.530500000004|
|2016|    6|         23599.374|
|2016|    7|         28608.259|
|2016|    8|        36818.3422|
+----+-----+------------------+
only showing top 20 rows



In [13]:
#loding data to s3

df_monthly_sales.write \
    .mode('overwrite') \
    .partitionBy('year') \
    .parquet('s3a://de-project-curated-abhi/monthly_sales/')

Top products by sales

In [9]:
top_products = df.groupBy('product_id', 'product_name') \
    .agg(sum('sales').alias('total_sales')) \
    .orderBy('total_sales', ascending=False)

top_products.show()

+---------------+--------------------+------------------+
|     product_id|        product_name|       total_sales|
+---------------+--------------------+------------------+
|TEC-CO-10004722|Canon imageCLASS ...| 61599.82400000001|
|OFF-BI-10003527|Fellowes PB500 El...|         27453.384|
|TEC-MA-10002412|Cisco TelePresenc...|          22638.48|
|FUR-CH-10002024|HON 5400 Series T...|         21870.576|
|OFF-BI-10001359|GBC DocuBind TL30...|         19823.479|
|OFF-BI-10000545|GBC Ibimaster 500...|           19024.5|
|TEC-CO-10001449|Hewlett Packard L...|         18839.686|
|TEC-MA-10001127|HP Designjet T520...|         18374.895|
|OFF-BI-10004995|GBC DocuBind P400...|         17965.068|
|OFF-SU-10000151|High Speed Automa...|17030.311999999998|
|TEC-MA-10000822|Lexmark MX611dhe ...|         16829.901|
|OFF-SU-10002881|Martin Yale Chadl...|           16656.2|
|OFF-BI-10001120|Ibico EPK-21 Elec...|15875.916000000001|
|FUR-BO-10004834|Riverside Palais ...|        15610.9656|
|TEC-MA-100010

In [14]:
#loding data to s3

top_products.write \
    .mode('overwrite') \
    .parquet('s3a://de-project-curated-abhi/top_products/')

Top costumers

In [10]:
df_top_customers = df.groupBy('customer_id', 'customer_name') \
    .agg(sum('sales').alias('total_sales')) \
    .orderBy('total_sales', ascending=False)

df_top_customers.show()

+-----------+------------------+------------------+
|customer_id|     customer_name|       total_sales|
+-----------+------------------+------------------+
|   SM-20320|       Sean Miller|          25043.05|
|   TC-20980|      Tamara Chand|19052.217999999997|
|   RB-19360|      Raymond Buch|         15117.339|
|   TA-21385|      Tom Ashbrook|          14595.62|
|   AB-10105|     Adrian Barton|14473.570999999998|
|   KL-16645|      Ken Lonsdale|         14175.229|
|   SC-20095|      Sanjit Chand|14142.333999999999|
|   HL-15040|      Hunter Lopez|12873.297999999999|
|   SE-20110|      Sanjit Engle|         12209.438|
|   CC-12370|Christopher Conant|         12129.072|
|   TS-21370|      Todd Sumrall|         11891.751|
|   GT-14710|         Greg Tran|11820.119999999999|
|   BM-11140|      Becky Martin|11789.630000000001|
|   SV-20365|       Seth Vernon|11470.949999999997|
|   CJ-12010|   Caroline Jumper|         11164.974|
|   CL-12565|       Clay Ludtke|         10880.546|
|   ME-17320

In [15]:
#loding data to s3

df_top_customers.write \
    .mode('overwrite') \
    .parquet('s3a://de-project-curated-abhi/top_customers/')